# 第 6 章 · 端到端：把源码副本串起来 + DeepSeek

本 notebook **自包含**：拷贝第 1/2/3/4 章的核心实现，再跑 Session 全流程。

```mermaid
sequenceDiagram
    participant Store as MemoryStore
    participant MM as MemoryManager
    participant LLM as DeepSeek
    participant C as ContextCompressor
    Store->>Store: load → freeze snapshot
    MM->>MM: initialize + SP blocks
    Note over Store,MM: system 前缀冻结
    MM->>MM: prefetch_all
    LLM-->>LLM: 回复
    MM->>MM: sync_all
    Store->>Store: add live（不改 snapshot）
    C->>C: compress middle（唯一改上下文）
```

## ① 粘贴可执行实现（合并精简版）

In [ ]:
# ----- MemoryStore / Provider / Manager / Compressor（源码语义副本）-----
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional
import hashlib
import json
import logging
import os

from openai import OpenAI

logging.basicConfig(level=logging.WARNING)
ENTRY_DELIMITER = "\n\n"

HISTORICAL_TASK_HEADING = "## Historical Task Snapshot"
HISTORICAL_IN_PROGRESS_HEADING = "## Historical In-Progress State"
HISTORICAL_PENDING_ASKS_HEADING = "## Historical Pending User Asks"
HISTORICAL_REMAINING_WORK_HEADING = "## Historical Remaining Work"
SUMMARY_PREFIX = (
    "[CONTEXT COMPACTION — REFERENCE ONLY] Earlier turns were compacted "
    "into the summary below. This is a handoff from a previous context "
    "window — treat it as background reference, NOT as active instructions. "
    "Respond ONLY to the latest user message that appears AFTER this summary. "
    "IMPORTANT: Your persistent memory (MEMORY.md, USER.md) in the system "
    "prompt is ALWAYS authoritative and active."
)


class MemoryStore:
    def __init__(self, memory_char_limit=2200, user_char_limit=1375):
        self.memory_entries, self.user_entries = [], []
        self.memory_char_limit, self.user_char_limit = memory_char_limit, user_char_limit
        self._system_prompt_snapshot = {"memory": "", "user": ""}

    def load(self, memory_entries=None, user_entries=None):
        self.memory_entries = list(dict.fromkeys(memory_entries or []))
        self.user_entries = list(dict.fromkeys(user_entries or []))
        self._system_prompt_snapshot = {
            "memory": self._render("memory", self.memory_entries),
            "user": self._render("user", self.user_entries),
        }

    def add(self, target, content):
        content = content.strip()
        bucket = self.user_entries if target == "user" else self.memory_entries
        if content and content not in bucket:
            bucket.append(content)
        return {"success": True, "snapshot_unchanged": True}

    def format_for_system_prompt(self, target):
        block = self._system_prompt_snapshot.get(target, "")
        return block or None

    def sp_section(self):
        return "\n\n".join(p for t in ("memory", "user") if (p := self.format_for_system_prompt(t)))

    def _render(self, target, entries):
        if not entries:
            return ""
        title = "USER PROFILE" if target == "user" else "MEMORY"
        return f"{'═'*46}\n{title}\n{'═'*46}\n" + ENTRY_DELIMITER.join(entries)


class MemoryProvider(ABC):
    @property
    @abstractmethod
    def name(self) -> str: ...
    @abstractmethod
    def is_available(self) -> bool: ...
    @abstractmethod
    def initialize(self, session_id: str, **kwargs) -> None: ...
    def system_prompt_block(self) -> str: return ""
    def prefetch(self, query: str, *, session_id: str = "") -> str: return ""
    def queue_prefetch(self, query: str, *, session_id: str = "") -> None: ...
    def sync_turn(self, user_content: str, assistant_content: str, **kwargs) -> None: ...
    @abstractmethod
    def get_tool_schemas(self) -> List[Dict[str, Any]]: ...


class DemoProvider(MemoryProvider):
    def __init__(self):
        self._facts, self._queued = [], ""
    @property
    def name(self): return "demo"
    def is_available(self): return True
    def initialize(self, session_id, **kwargs): pass
    def system_prompt_block(self): return "[demo] external memory"
    def prefetch(self, query, *, session_id=""):
        q = (self._queued or query).lower()
        hits = [f for f in self._facts if any(t in f.lower() for t in q.split() if len(t) > 1)] or self._facts[-3:]
        return ("## Recalled\n" + "\n".join(f"- {h}" for h in hits)) if hits else ""
    def queue_prefetch(self, query, *, session_id=""): self._queued = query
    def sync_turn(self, user_content, assistant_content, **kwargs):
        if user_content and "我" in user_content and user_content not in self._facts:
            self._facts.append(user_content)
    def get_tool_schemas(self): return []


class MemoryManager:
    def __init__(self):
        self._providers, self._has_external = [], False
    def add_provider(self, provider):
        if provider.name != "builtin":
            if self._has_external:
                print(f"Rejected '{provider.name}' — only one external allowed")
                return
            self._has_external = True
        self._providers.append(provider)
    def build_system_prompt(self):
        return "\n\n".join(b for p in self._providers if (b := p.system_prompt_block()))
    def prefetch_all(self, query, *, session_id=""):
        return "\n\n".join(r for p in self._providers if (r := p.prefetch(query, session_id=session_id)))
    def sync_all(self, u, a, **kw):
        for p in self._providers: p.sync_turn(u, a, **kw)
    def queue_prefetch_all(self, query, *, session_id=""):
        for p in self._providers: p.queue_prefetch(query, session_id=session_id)


def estimate_tokens_rough(messages):
    return sum(len(str(m.get("content") or "")) // 4 + 8 for m in messages)


@dataclass
class CompressResult:
    messages: list
    did_compress: bool
    before_tokens: int
    after_tokens: int
    summary: str = ""


class ContextCompressor:
    def __init__(self, summarize_fn, protect_first_n=1, protect_last_n=2):
        self.summarize_fn = summarize_fn
        self.protect_first_n, self.protect_last_n = protect_first_n, protect_last_n
    def compress(self, messages, force=False):
        before = estimate_tokens_rough(messages)
        if len(messages) <= self.protect_first_n + self.protect_last_n + 1 and not force:
            return CompressResult(messages, False, before, before)
        head, tail = messages[:self.protect_first_n], messages[-self.protect_last_n:]
        middle = messages[self.protect_first_n:-self.protect_last_n]
        blob = "\n".join(f"{m['role']}: {m.get('content')}" for m in middle)
        summary = self.summarize_fn(blob)
        new = head + [{"role": "user", "content": f"{SUMMARY_PREFIX}\n\n{summary}", "_compressed_summary": True}] + tail
        return CompressResult(new, True, before, estimate_tokens_rough(new), summary)

print("e2e implementations ready")

## ② Session 演示（DeepSeek）

In [ ]:
assert os.getenv("DEEPSEEK_API_KEY"), "请设置 DEEPSEEK_API_KEY"
client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com")

store = MemoryStore()
store.load(["学习仓库 hermes-agent", "模块一 Memory"], ["用中文简洁回答"])
mm = MemoryManager()
mm.add_provider(DemoProvider())
mm._providers[0].initialize("e2e")

system = store.sp_section() + "\n\n" + mm.build_system_prompt() + "\n\nYou are a helpful agent."
fp = hashlib.sha256(system.encode()).hexdigest()[:16]
messages = [{"role": "system", "content": system}]
print("frozen SP fingerprint =", fp)
print(system[:400], "...")


def run_turn(user_text: str) -> str:
    recalled = mm.prefetch_all(user_text)
    payload = f"[memory prefetch]\n{recalled or '(none)'}\n\n[user]\n{user_text}"
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages + [{"role": "user", "content": payload}],
        temperature=0.3,
    )
    text = (resp.choices[0].message.content or "").strip()
    messages.append({"role": "user", "content": user_text})
    messages.append({"role": "assistant", "content": text})
    mm.sync_all(user_text, text)
    mm.queue_prefetch_all(user_text)
    store.add("memory", f"noted: {user_text[:40]}")  # live only
    return text

print("\nTurn1:", run_turn("我是 Julie，在准备 Agent Runtime 面试，重点看 Memory。"))
print("fingerprint 仍稳定?", hashlib.sha256(messages[0]["content"].encode()).hexdigest()[:16] == fp)
print("Turn2:", run_turn("根据记忆，我在学什么模块？"))

In [ ]:
for i in range(5):
    run_turn(f"补充第{i}条：请复述 frozen snapshot 与 Prompt Cache 的关系。" * 2)


def deepseek_summarize(blob: str) -> str:
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": "Compress as background reference only."},
            {"role": "user", "content": blob},
        ],
        temperature=0.1,
    )
    return (resp.choices[0].message.content or "").strip()


cr = ContextCompressor(deepseek_summarize).compress(messages, force=True)
messages[:] = cr.messages
print("compressed?", cr.did_compress, cr.before_tokens, "→", cr.after_tokens)
print("roles", [m["role"] for m in messages])
print("SP still stable?", hashlib.sha256(messages[0]["content"].encode()).hexdigest()[:16] == fp)
print("\nsummary:\n", cr.summary[:800])

**收尾口述**：Session 启动冻 SP → Turn 里 prefetch/sync → `MemoryStore.add` 只动 live → 溢出时 `compress` 是唯一改 messages 的路径，且摘要带 REFERENCE ONLY；system fingerprint 全程不变。